# Driver Ranking Analysis


In [6]:
%pip install pandas
%pip install numpy

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Load and Prepare Data

In [8]:
# Load the lap-weather dataset from Luis
df = pd.read_csv('../data/lap_weather_data_2018_2025.csv')

print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:")
print(df.dtypes)

Data shape: (189248, 42)

Columns: ['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'FastF1Generated', 'IsAccurate', 'Location', 'Year', 'EventName', 'LapStartTimeUTC', 'IsPitLap', 'IsTerminalLap', 'AirTemp', 'Humidity', 'Pressure', 'TrackTemp', 'WindDirection', 'WindSpeed', 'Rainfall']

Data types:
Time                   object
Driver                 object
DriverNumber            int64
LapTime                object
LapNumber             float64
Stint                 float64
PitOutTime             object
PitInTime              object
Sector1Time            object
Sector2Time            object
Sector3Time            object
Sector1SessionTime     object
Sector2SessionTi

In [ ]:
# Check data quality
print(f"Unique races: {df['EventName'].nunique()}")
print(f"Unique drivers: {df['Driver'].nunique()}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")
print(f"\nPosition column null count: {df['Position'].isna().sum()} / {len(df)}")
print(f"Position data type: {df['Position'].dtype}")
print(df[['Driver', 'EventName', 'Year', 'LapNumber', 'Position', 'Team']].head(10))

Unique races: 36
Unique drivers: 43
Year range: 2018 - 2025

Position column null count: 262 / 189248
Position data type: float64

Sample data:
  Driver              EventName  Year  LapNumber  Position          Team
0    GAS  Australian Grand Prix  2018        1.0      17.0  Racing Bulls
1    GAS  Australian Grand Prix  2018        2.0      17.0  Racing Bulls
2    GAS  Australian Grand Prix  2018        3.0      17.0  Racing Bulls
3    GAS  Australian Grand Prix  2018        4.0      17.0  Racing Bulls
4    GAS  Australian Grand Prix  2018        5.0      17.0  Racing Bulls
5    GAS  Australian Grand Prix  2018        6.0      16.0  Racing Bulls
6    GAS  Australian Grand Prix  2018        7.0      16.0  Racing Bulls
7    GAS  Australian Grand Prix  2018        8.0      16.0  Racing Bulls
8    GAS  Australian Grand Prix  2018        9.0      16.0  Racing Bulls
9    GAS  Australian Grand Prix  2018       10.0      16.0  Racing Bulls


In [10]:
# cast Position as numeric
df['Position'] = pd.to_numeric(df['Position'], errors='coerce')

# unique race identifier
df['RaceID'] = df['EventName'] + ' ' + df['Year'].astype(str)

print(f"Races with data: {df['RaceID'].nunique()}")
print(f"\nSample races:")
print(df['RaceID'].unique()[:5])

Races with data: 172

Sample races:
['Australian Grand Prix 2018' 'Bahrain Grand Prix 2018'
 'Chinese Grand Prix 2018' 'Azerbaijan Grand Prix 2018'
 'Spanish Grand Prix 2018']


## Create Race-Level Summary

In [21]:
race_results_list = []

for race_id in df['RaceID'].unique():
    race_data = df[df['RaceID'] == race_id]
    
    # Get unique drivers in this race
    drivers = race_data[['Driver', 'DriverNumber', 'Team']].drop_duplicates()
    
    for _, driver_row in drivers.iterrows():
        driver = driver_row['Driver']
        driver_race = race_data[race_data['Driver'] == driver].copy()
        
        if len(driver_race) == 0:
            continue
            
        # Extract race info
        event_name = race_data['EventName'].iloc[0]
        year = race_data['Year'].iloc[0]
        location = race_data['Location'].iloc[0]
        
        # Position tracking
        valid_positions = driver_race['Position'].dropna()
        
        if len(valid_positions) == 0:
            continue
        
        # Grid position (first lap position)
        first_lap = driver_race[driver_race['LapNumber'] == 1.0]
        grid_pos = first_lap['Position'].iloc[0] if len(first_lap) > 0 and pd.notna(first_lap['Position'].iloc[0]) else np.nan
        
        # Finishing position (last valid position)
        finishing_pos = valid_positions.iloc[-1]
        
        # Position statistics
        best_pos = valid_positions.min()
        worst_pos = valid_positions.max()
        avg_pos = valid_positions.mean()
        pos_std = valid_positions.std()
        
        # Positions gained/lost 
        if pd.notna(grid_pos):
            positions_gained = grid_pos - finishing_pos
        else:
            positions_gained = np.nan
        
        # Laps led
        laps_led = len(driver_race[driver_race['Position'] == 1.0])
        
        # Pit stops
        pit_stops = len(driver_race[driver_race['IsPitLap'] == True])
        
        # DNF detection 
        max_laps_any_driver = race_data['LapNumber'].max()
        driver_max_laps = driver_race['LapNumber'].max()
        dnf = driver_max_laps < (max_laps_any_driver - 2)  # Allow 2 lap tolerance
        
        race_results_list.append({
            'Driver': driver,
            'DriverNumber': driver_row['DriverNumber'],
            'Team': driver_row['Team'],
            'EventName': event_name,
            'Year': year,
            'Location': location,
            'GridPosition': grid_pos,
            'FinishingPosition': finishing_pos,
            'BestPosition': best_pos,
            'WorstPosition': worst_pos,
            'AvgPosition': avg_pos,
            'PositionStdDev': pos_std,
            'PositionsGained': positions_gained,
            'LapsLed': laps_led,
            'PitStops': pit_stops,
            'LapsCompleted': int(driver_max_laps),
            'DNF': dnf
        })

race_results_df = pd.DataFrame(race_results_list)
print(f"race_results_df: {race_results_df.shape}")
print(f"\nFirst race:")
print(race_results_df.head(19))

race_results_df: (3349, 17)

First race:
   Driver  DriverNumber             Team              EventName  Year  \
0     GAS            10     Racing Bulls  Australian Grand Prix  2018   
1     PER            11     Aston Martin  Australian Grand Prix  2018   
2     ALO            14          McLaren  Australian Grand Prix  2018   
3     LEC            16      Kick Sauber  Australian Grand Prix  2018   
4     STR            18         Williams  Australian Grand Prix  2018   
5     VAN             2          McLaren  Australian Grand Prix  2018   
6     MAG            20             Haas  Australian Grand Prix  2018   
7     HUL            27           Alpine  Australian Grand Prix  2018   
8     HAR            28     Racing Bulls  Australian Grand Prix  2018   
9     RIC             3  Red Bull Racing  Australian Grand Prix  2018   
10    OCO            31     Aston Martin  Australian Grand Prix  2018   
11    VER            33  Red Bull Racing  Australian Grand Prix  2018   
12    SIR 

In [23]:
print("race_results_df summary:")
print(race_results_df.describe())
print(f"\nDNF count: {race_results_df['DNF'].sum()}")
print(f"Races with drivers leading: {race_results_df[race_results_df['LapsLed'] > 0].shape[0]}")

race_results_df summary:
       DriverNumber         Year  GridPosition  FinishingPosition  \
count   3349.000000  3349.000000   3349.000000        3349.000000   
mean      27.976112  2021.663780     10.257390           9.689460   
std       24.608694     2.303536      5.649725           5.318669   
min        1.000000  2018.000000      1.000000           1.000000   
25%       10.000000  2020.000000      5.000000           5.000000   
50%       20.000000  2022.000000     10.000000          10.000000   
75%       44.000000  2024.000000     15.000000          14.000000   
max       99.000000  2025.000000     20.000000          20.000000   

       BestPosition  WorstPosition  AvgPosition  PositionStdDev  \
count   3349.000000    3349.000000  3349.000000     3335.000000   
mean       7.122723      13.339504     9.928888        1.685339   
std        4.540030       5.537290     5.091808        1.016996   
min        1.000000       1.000000     1.000000        0.000000   
25%        3.00000

## Create Driver-Level Statistics 

In [29]:
#statistics per driver across all races participated in
driver_stats_list = []

for driver in race_results_df['Driver'].unique():
    driver_races = race_results_df[race_results_df['Driver'] == driver]
    
    # driver info
    driver_num = driver_races['DriverNumber'].iloc[0]
    team = driver_races['Team'].iloc[0]
    
    # race counts
    races_entered = len(driver_races)
    races_completed = len(driver_races[driver_races['DNF'] == False])
    dnf_count = races_entered - races_completed
    
    # Position statistics
    avg_grid_pos = driver_races['GridPosition'].mean()
    avg_finish_pos = driver_races['FinishingPosition'].mean()
    
    # Position gained/lost
    avg_positions_gained = driver_races['PositionsGained'].mean()
    races_with_gains = len(driver_races[driver_races['PositionsGained'] > 0])
    races_with_losses = len(driver_races[driver_races['PositionsGained'] < 0])
    
    # Driver Performance metrics
    podiums = len(driver_races[driver_races['FinishingPosition'] <= 3])
    poles = len(driver_races[driver_races['GridPosition'] == 1.0])
    wins = len(driver_races[driver_races['FinishingPosition'] == 1.0])
    total_laps_led = driver_races['LapsLed'].sum()
    
    # DriverConsistency
    avg_position_std = driver_races['PositionStdDev'].mean()
    
    driver_stats_list.append({
        'Driver': driver,
        'DriverNumber': driver_num,
        'Team': team,
        'RacesEntered': races_entered,
        'RacesCompleted': races_completed,
        'DNFCount': dnf_count,
        'DNFRate': dnf_count / races_entered if races_entered > 0 else 0,
        'AvgGridPosition': avg_grid_pos,
        'AvgFinishPosition': avg_finish_pos,
        'AvgPositionsGained': avg_positions_gained,
        'RacesWithGains': races_with_gains,
        'RacesWithLosses': races_with_losses,
        'Podiums': podiums,
        'Poles': poles,
        'Wins': wins,
        'TotalLapsLed': total_laps_led,
        'AvgPositionConsistency': avg_position_std
    })

driver_stats_df = pd.DataFrame(driver_stats_list).sort_values('RacesEntered', ascending=False)
print(f"Driver Statistics DataFrame shape: {driver_stats_df.shape}")
print(f"\nTop 10 drivers (2018-2025):")
print(driver_stats_df.head(10))

Driver Statistics DataFrame shape: (43, 17)

Top 10 drivers (2018-2025):
   Driver  DriverNumber             Team  RacesEntered  RacesCompleted  \
13    HAM            44         Mercedes           168             161   
0     GAS            10     Racing Bulls           166             147   
11    VER            33  Red Bull Racing           166             154   
3     LEC            16      Kick Sauber           164             145   
4     STR            18         Williams           164             145   
15    SAI            55           Alpine           164             147   
22    NOR             4          McLaren           152             139   
23    RUS            63         Williams           151             135   
17    BOT            77         Mercedes           147             129   
10    OCO            31     Aston Martin           145             127   

    DNFCount   DNFRate  AvgGridPosition  AvgFinishPosition  \
13         7  0.041667         5.250000           

In [27]:
# Show drivers with most position gains
print("\nTop 10 drivers by average positions gained per race:")
print(driver_stats_df.nlargest(10, 'AvgPositionsGained')[['Driver', 'Team', 'AvgPositionsGained', 'RacesEntered']])

print("\nMost successful drivers (by wins):")
print(driver_stats_df.nlargest(10, 'Wins')[['Driver', 'Team', 'Wins', 'Podiums', 'Poles']])


Top 10 drivers by average positions gained per race:
   Driver          Team  AvgPositionsGained  RacesEntered
5     VAN       McLaren            3.500000            20
21    KVY  Racing Bulls            2.368421            38
1     PER  Aston Martin            2.146853           143
19    ERI   Kick Sauber            2.100000            20
33    DEV      Williams            1.818182            11
37    BEA       Ferrari            1.629630            27
8     HAR  Racing Bulls            1.578947            19
32    ZHO   Kick Sauber            1.402985            67
16    RAI       Ferrari            1.115385            78
26    LAT      Williams            1.083333            60

Most successful drivers (by wins):
   Driver             Team  Wins  Podiums  Poles
11    VER  Red Bull Racing    70      119     57
13    HAM         Mercedes    40       88     28
22    NOR          McLaren    11       45     10
3     LEC      Kick Sauber     9       51     20
35    PIA          McLaren 

## Create Lap-by-Lap Position Tracking

In [30]:
# Create a pivot table for lap-by-lap position tracking

position_tracking_list = []

for race_id in df['RaceID'].unique():
    race_data = df[df['RaceID'] == race_id]
    event_name = race_data['EventName'].iloc[0]
    year = race_data['Year'].iloc[0]
    
    race_pivot = race_data.pivot_table(
        index=['Driver', 'Team', 'DriverNumber'],
        columns='LapNumber',
        values='Position',
        aggfunc='first'
    )
    
    # race identifiers
    race_pivot['EventName'] = event_name
    race_pivot['Year'] = year
    race_pivot['RaceID'] = race_id
    
    position_tracking_list.append(race_pivot)

print(f"\nFirst race (first 5 drivers):")
print(position_tracking_list[0].head(5))


First race (first 5 drivers):
LapNumber                          1.0   2.0   3.0   4.0   5.0   6.0   7.0  \
Driver Team         DriverNumber                                             
ALO    McLaren      14            10.0  10.0  10.0  10.0  10.0  10.0  10.0   
BOT    Mercedes     77            15.0  15.0  15.0  14.0  14.0  14.0  14.0   
ERI    Kick Sauber  9             16.0  16.0  16.0  16.0  16.0  18.0   NaN   
GAS    Racing Bulls 10            17.0  17.0  17.0  17.0  17.0  16.0  16.0   
GRO    Haas         8              6.0   6.0   6.0   6.0   6.0   6.0   6.0   

LapNumber                          8.0   9.0  10.0  ...  52.0  53.0  54.0  \
Driver Team         DriverNumber                    ...                     
ALO    McLaren      14            10.0  10.0  10.0  ...   5.0   5.0   5.0   
BOT    Mercedes     77            14.0  13.0  13.0  ...   8.0   8.0   8.0   
ERI    Kick Sauber  9              NaN   NaN   NaN  ...   NaN   NaN   NaN   
GAS    Racing Bulls 10            16.

## Position Change Analysis

In [32]:
# dataframe tracking position changes within races
position_changes_list = []

for race_id in df['RaceID'].unique():
    race_data = df[df['RaceID'] == race_id]
    event_name = race_data['EventName'].iloc[0]
    year = race_data['Year'].iloc[0]
    location = race_data['Location'].iloc[0]
    
    for driver in race_data['Driver'].unique():
        driver_data = race_data[race_data['Driver'] == driver].sort_values('LapNumber')
        
        # Remove rows with NaN 
        driver_data = driver_data[driver_data['Position'].notna()]
        
        if len(driver_data) < 2:
            continue
        
        # Get driver info
        team = driver_data['Team'].iloc[0]
        driver_num = driver_data['DriverNumber'].iloc[0]
        
        # Track position changes lap by lap
        positions = driver_data['Position'].values
        lap_numbers = driver_data['LapNumber'].values
        
        # Calculate position changes
        for i in range(1, len(positions)):
            position_change = positions[i-1] - positions[i]  
            
            position_changes_list.append({
                'Driver': driver,
                'DriverNumber': driver_num,
                'Team': team,
                'EventName': event_name,
                'Year': year,
                'Location': location,
                'RaceID': race_id,
                'FromLap': int(lap_numbers[i-1]),
                'ToLap': int(lap_numbers[i]),
                'PositionBefore': positions[i-1],
                'PositionAfter': positions[i],
                'PositionChange': position_change
            })

position_changes_df = pd.DataFrame(position_changes_list)
print(f"position_changes_df: {position_changes_df.shape}")
print(f"\nFirst 10 position changes:")
print(position_changes_df.head(10))

position_changes_df: (185637, 12)

First 10 position changes:
  Driver  DriverNumber          Team              EventName  Year   Location  \
0    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
1    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
2    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
3    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
4    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
5    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
6    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
7    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
8    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   
9    GAS            10  Racing Bulls  Australian Grand Prix  2018  Melbourne   

                       RaceID  FromLap  ToLap  PositionBe

In [17]:
print("\nMost dramatic position gains in single lap:")
print(position_changes_df.nlargest(10, 'PositionChange')[['Driver', 'EventName', 'Year', 'FromLap', 'PositionChange']])

print("\nMost dramatic position losses in single lap:")
print(position_changes_df.nsmallest(10, 'PositionChange')[['Driver', 'EventName', 'Year', 'FromLap', 'PositionChange']])

print(f"\nAverage position changes per lap: {position_changes_df['PositionChange'].mean():.3f}")
print(f"Median position changes per lap: {position_changes_df['PositionChange'].median():.3f}")


Most dramatic position gains in single lap:
       Driver             EventName  Year  FromLap  PositionChange
71412     STR    Styrian Grand Prix  2021        4            13.0
72689     STR   Austrian Grand Prix  2021        4            13.0
123385    ZHO      Dutch Grand Prix  2023        2            13.0
74496     HAM  Hungarian Grand Prix  2021        3            12.0
122835    GAS      Dutch Grand Prix  2023        2            12.0
47236     MAG  Hungarian Grand Prix  2020        3            11.0
123048    LEC      Dutch Grand Prix  2023        2            11.0
122906    PER      Dutch Grand Prix  2023        2            10.0
47644     GRO  Hungarian Grand Prix  2020        3             9.0
52476     LAT    Italian Grand Prix  2020       22             9.0

Most dramatic position losses in single lap:
       Driver              EventName  Year  FromLap  PositionChange
135321    VER  Australian Grand Prix  2024        3           -17.0
38546     LEC    Japanese Grand Prix

## Exporting Dataframes

In [ ]:
# Save the dataframes to CSV 
race_results_df.to_csv('../data/race_results_summary.csv', index=False)
driver_stats_df.to_csv('../data/driver_statistics.csv', index=False)
position_changes_df.to_csv('../data/position_changes.csv', index=False)


print(f"  - race_results_df: {race_results_df.shape}")
print(f"  - driver_stats_df: {driver_stats_df.shape}")
print(f"  - position_changes_df: {position_changes_df.shape}")

  - race_results_df: (3349, 17)
  - driver_stats_df: (43, 17)
  - position_changes_df: (185637, 12)
